# 2. Data Cleaning and Preprocessing

## Objective
This notebook performs data cleaning and preprocessing on raw intraday
NIFTY futures and options data.

Steps include:
- Handling missing values
- Data type normalization
- Futures contract rollover handling
- ATM strike calculation
- Timestamp alignment
- Merging futures and options datasets

Output:
- Cleaned and merged intraday dataset


In [1]:
import pandas as pd
import numpy as np

In [5]:
futures = pd.read_csv("../data/raw/nifty_futures_intraday.csv", low_memory=False)
options_ce = pd.read_csv("../data/raw/nifty_options_ce_intraday.csv", low_memory=False)
options_pe = pd.read_csv("../data/raw/nifty_options_pe_intraday.csv", low_memory=False)

In [7]:
print("Futures columns:\n", futures.columns.tolist())
print("\nOptions CE columns:\n", options_ce.columns.tolist())
print("\nOptions PE columns:\n", options_pe.columns.tolist())

Futures columns:
 ['symbol', 'date', 'expiry', 'open', 'high', 'low', 'close', 'ltp', 'settle_price', 'contracts', 'turnover__in___rs_lakhs', 'open_interest', 'change_in_oi', 'underlying_value', 'timestamp']

Options CE columns:
 ['symbol', 'date', 'expiry', 'option_type', 'strike', 'open', 'high', 'low', 'close', 'ltp', 'settle_price', 'contracts', 'turnover__in__rs_lakhs', 'premium_turnover__in___rs_lakhs', 'open_interest', 'change_in_oi', 'underlying_value', 'timestamp']

Options PE columns:
 ['symbol', 'date', 'expiry', 'option_type', 'strike', 'open', 'high', 'low', 'close', 'ltp', 'settle_price', 'contracts', 'turnover__in__rs_lakhs', 'premium_turnover__in___rs_lakhs', 'open_interest', 'change_in_oi', 'underlying_value', 'timestamp']


In [8]:
def clean_columns(df):
    df.columns = (
        df.columns
          .str.strip()
          .str.replace(r"\s+", " ", regex=True)
    )
    return df

futures = clean_columns(futures)
options_ce = clean_columns(options_ce)
options_pe = clean_columns(options_pe)

In [13]:
numeric_cols_fut = [
    "open", "high", "low", "close",
    "open_interest", "change_in_oi"
]

numeric_cols_opt = [
    "open", "high", "low", "close",
    "strike", "open_interest",
    "change_in_oi", "contracts"
]

for col in numeric_cols_fut:
    futures[col] = pd.to_numeric(futures[col], errors="coerce")

for col in numeric_cols_opt:
    options_ce[col] = pd.to_numeric(options_ce[col], errors="coerce")
    options_pe[col] = pd.to_numeric(options_pe[col], errors="coerce")

In [14]:
for df in [futures, options_ce, options_pe]:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df.sort_values("timestamp", inplace=True)

In [16]:
futures = futures.dropna(subset=["timestamp", "close", "open_interest"])
options_ce = options_ce.dropna(subset=["timestamp", "strike", "close", "open_interest"])
options_pe = options_pe.dropna(subset=["timestamp", "strike", "close", "open_interest"])

In [17]:
futures["expiry"] = pd.to_datetime(futures["expiry"])

futures = (
    futures.sort_values(["timestamp", "expiry"])
           .groupby("timestamp", as_index=False)
           .first()
)

In [18]:
def nearest_strike(price, strikes):
    return min(strikes, key=lambda x: abs(x - price))

strike_map = (
    options_ce.groupby("timestamp")["strike"]
              .unique()
              .to_dict()
)

futures["atm_strike"] = futures.apply(
    lambda row: nearest_strike(
        row["close"],
        strike_map.get(row["timestamp"], [])
    ),
    axis=1
)

In [19]:
STRIKE_GAP = 50

def filter_atm_range(df, atm_df):
    df = df.merge(
        atm_df[["timestamp", "atm_strike"]],
        on="timestamp",
        how="inner"
    )
    return df[
        df["strike"].between(
            df["atm_strike"] - 2 * STRIKE_GAP,
            df["atm_strike"] + 2 * STRIKE_GAP
        )
    ]

options_ce = filter_atm_range(options_ce, futures)
options_pe = filter_atm_range(options_pe, futures)

In [20]:
merged = (
    futures
    .merge(options_ce, on="timestamp", suffixes=("_fut", "_ce"))
    .merge(options_pe, on="timestamp", suffixes=("", "_pe"))
)

In [21]:
merged.to_csv("../data/cleaned/nifty_merged_intraday.csv", index=False)
print("Saved cleaned dataset:", merged.shape)

Saved cleaned dataset: (299248, 52)
